In [1]:
import pygame
import random
import math
import numpy as np
import matplotlib.pyplot as plt
from class_function import *

pygame 2.6.1 (SDL 2.28.4, Python 3.11.4)
Hello from the pygame community. https://www.pygame.org/contribute.html


In [2]:
WIDTH, HEIGHT = 1000, 700
FPS = 60
FOV = 500

BLACK = np.array([10, 10, 15])
WHITE = np.array([255, 255, 255])
FOLDER_COLOR = np.array([100, 200, 255]) # Bleu fluo pour les dossiers
FILE_COLOR = np.array([200, 255, 100])   # Vert fluo pour les fichiers/sous-dossiers
LINE_COLOR = np.array([50, 80, 120])

In [3]:
pygame.init()
screen = pygame.display.set_mode((WIDTH, HEIGHT))
clock = pygame.time.Clock()
running = True

pygame.font.init()
font = pygame.font.SysFont(None, 12)


#bureau_init = Bureau3D([Folder3D("Folder1", 300, 100, 0, None, [Folder3D("Folder3", 200, 350, 100, None, []), Folder3D("Folder4", 250, 400, 50, None, []), Folder3D("Folder5", 100, 300, 100, None, [])], is_open=True), Folder3D("Folder2", 500, 200, 100, None, [], is_open=True)])  # Génère une structure de dossiers en forme de sphère
#bureau_init = generate_spheric_bureau([10, 5, 2], [0, 0, 0], 200)
chemin_bureau = r"C:\Users\barna\Documents"

bureau_init = generer_depuis_mon_bureau(chemin_bureau, (0, 0, 0), 300, profondeur_max=4)
for folder in bureau_init.folders:
    folder.is_open = True  # Ouvre tous les dossiers par défaut
camera = Camera(0, 0, -FOV)
afficher_popup = False

while running:
    # 1. PRÉCALCUL DE LA CAMÉRA
    cam_trig = {
        'cos_x': math.cos(camera.angle_x),
        'sin_x': math.sin(camera.angle_x),
        'cos_y': math.cos(camera.angle_y),
        'sin_y': math.sin(camera.angle_y)
    }
    
    # =========================================================
    # 2. GESTION DES ÉVÉNEMENTS (Actions uniques / Interface)
    # =========================================================
    for event in pygame.event.get():
            
        if event.type == pygame.KEYDOWN:
            if afficher_popup:
                # Fermer le popup (Entrée ou Échap)
                if event.key == pygame.K_RETURN or event.key == pygame.K_ESCAPE:
                    afficher_popup = False
                    
                # Action "O" : Afficher/Cacher les enfants
                elif event.key == pygame.K_o:
                    if nearest_folder and nearest_folder.children:
                        for child in nearest_folder.children:
                            child.is_open = not child.is_open
                    afficher_popup = False  # Ferme le menu après avoir modifié les dossiers
                elif event.key == pygame.K_d:
                    nearest_folder.ouvrir_sur_ordinateur()
            else: # Si on est en mode navigation (pas de popup)
                if event.key == pygame.K_RETURN:
                    nearest_folder, _ = bureau_init.find_nearest_folder(camera, cam_trig)
                    if nearest_folder:
                        nom_dossier_cible = nearest_folder.name
                        afficher_popup = True
                if event.key == pygame.K_ESCAPE:
                    running = False
    # =========================================================
    # 3. DÉPLACEMENTS CONTINUS (Bloqués si le popup est ouvert)
    # =========================================================
    keys = pygame.key.get_pressed()

    if not afficher_popup:
        dx_avant = -math.sin(camera.angle_y) * math.cos(camera.angle_x)
        dy_avant = math.sin(camera.angle_x)
        dz_avant = math.cos(camera.angle_y) * math.cos(camera.angle_x)
        vitesse = 8
        vitesse_rot = 0.03

        if keys[pygame.K_z]:
            camera.x += dx_avant * vitesse; camera.y += dy_avant * vitesse; camera.z += dz_avant * vitesse
        if keys[pygame.K_s]:
            camera.x -= dx_avant * vitesse; camera.y -= dy_avant * vitesse; camera.z -= dz_avant * vitesse

        if keys[pygame.K_LEFT]: camera.angle_y += vitesse_rot
        if keys[pygame.K_RIGHT]: camera.angle_y -= vitesse_rot
        if keys[pygame.K_UP]: camera.angle_x -= vitesse_rot
        if keys[pygame.K_DOWN]: camera.angle_x += vitesse_rot

    # =========================================================
    # 4. RENDU 3D (S'exécute à chaque frame !)
    # =========================================================
    pixel_buffer = np.zeros((HEIGHT, WIDTH, 3), dtype=np.float32)
    labels_to_draw = []

    for folder in bureau_init.folders:
        if folder.is_open:
            pixel_buffer, _, _ = folder.draw_folder(pixel_buffer, camera, labels_to_draw, cam_trig)
             
    pixel_buffer = draw_pointer(pixel_buffer)
    pixel_buffer = np.clip(pixel_buffer, 0, 255).astype(np.uint8)
    pygame_buffer = np.swapaxes(pixel_buffer, 0, 1)
    surface = pygame.surfarray.make_surface(pygame_buffer)
    
    screen.blit(surface, (0, 0))

    # Dessin des labels 3D par dessus les étoiles
    for label, x, y in labels_to_draw:
        text_surface = font.render(label, True, (255, 255, 255))
        screen.blit(text_surface, (x, y))

    # =========================================================
    # 5. RENDU DU POPUP (S'affiche par dessus le rendu 3D)
    # =========================================================
    if afficher_popup:
        voile = pygame.Surface((WIDTH, HEIGHT), pygame.SRCALPHA)
        voile.fill((0, 0, 0, 180)) # Assombrit la 3D derrière
        screen.blit(voile, (0, 0))

        largeur_box, hauteur_box = 700, 150
        x_box = (WIDTH - largeur_box) // 2
        y_box = (HEIGHT - hauteur_box) // 2
        
        pygame.draw.rect(screen, (30, 40, 60), (x_box, y_box, largeur_box, hauteur_box), border_radius=15)
        pygame.draw.rect(screen, (255, 255, 255), (x_box, y_box, largeur_box, hauteur_box), width=3, border_radius=15)

        font_titre = pygame.font.SysFont(None, 40)
        font_sous_titre = pygame.font.SysFont(None, 24)
        
        texte_dossier = font_titre.render(nom_dossier_cible, True, (255, 255, 255))
        
        # On sépare les instructions sur deux lignes pour éviter le débordement
        texte_fermer_1 = font_sous_titre.render("Appuyez sur O pour enlever/ajouter les dossiers connexes", True, (150, 150, 150))
        texte_fermer_2 = font_sous_titre.render("Appuyez sur D pour ouvrir le dossier.", True, (150, 150, 150))
        
        # On ajuste l'espacement vertical (Y) pour faire de la place aux 3 lignes
        screen.blit(texte_dossier, texte_dossier.get_rect(center=(WIDTH // 2, HEIGHT // 2 - 25)))
        screen.blit(texte_fermer_1, texte_fermer_1.get_rect(center=(WIDTH // 2, HEIGHT // 2 + 15)))
        screen.blit(texte_fermer_2, texte_fermer_2.get_rect(center=(WIDTH // 2, HEIGHT // 2 + 40)))
        
    pygame.display.flip()
    clock.tick(FPS)

    

[Génération] Lecture de la racine OK : 26 éléments trouvés.
[Erreur sur un sous-dossier ignoré] C:\Users\barna\Documents\Ma musique -> [WinError 5] Accès refusé: 'C:\\Users\\barna\\Documents\\Ma musique'
[Erreur sur un sous-dossier ignoré] C:\Users\barna\Documents\Mes images -> [WinError 5] Accès refusé: 'C:\\Users\\barna\\Documents\\Mes images'
[Erreur sur un sous-dossier ignoré] C:\Users\barna\Documents\Mes vidéos -> [WinError 5] Accès refusé: 'C:\\Users\\barna\\Documents\\Mes vidéos'
Ouverture de : C:\Users\barna\Documents\Administratif
Ouverture de : C:\Users\barna\Documents\Mes images
Ouverture de : C:\Users\barna\Documents\Administratif\Passeport-rogné_Haut.pdf
Ouverture de : C:\Users\barna\Documents\Mec stylé.png
